# CIFAR-10 small ViT — audited optimizer baselines

This notebook runs the same six-block Vision Transformer with SGD + Nesterov, AdamW, and Muon + auxiliary AdamW. The reference recipe uses a fixed 45k/5k train/validation split, protects the official test set from model selection, and applies the full DeiT-style regularization stack: RandAugment, color jitter, mixup, CutMix, label smoothing, random erasing, stochastic depth, gradient clipping, optimizer-specific warm-up, cosine decay, and non-zero LR floors.

The committed default is 300 epochs and three seeds. Checkpoints are restartable. WeightWatcher runs on CPU matrix copies with `ERG=True, randomize=True` and requires direct `alpha`, `ERG_gap`, and `num_traps` columns.


In [ ]:
from pathlib import Path
import os, sys, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir():
        ROOT = candidate
        break
    if (path / 'rg_baselines').is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError('Run from a clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rg_baselines.vit_cifar10 import (
    DEFAULT_VIT_SEEDS, SmallViT, ViTBaselineConfig,
    choose_device, muon_parameter_names, run_vit_baseline, summarize_final,
)

def resolve_dir(name, default):
    raw = os.environ.get(name)
    path = Path(raw).expanduser() if raw else default
    return (path if path.is_absolute() else Path.cwd() / path).resolve()

RUN_ROOT = resolve_dir('RG_BASELINE_RUN_ROOT', ROOT / 'runs')
DATA_DIR = resolve_dir('RG_BASELINE_DATA_DIR', ROOT / 'data')
EXPERIMENT_ROOT = RUN_ROOT / 'cifar10_vit'
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = choose_device()
print('device:', DEVICE)


In [ ]:
CONFIG = ViTBaselineConfig()
SEEDS = DEFAULT_VIT_SEEDS
model = SmallViT(CONFIG)
print(f'parameters: {sum(p.numel() for p in model.parameters()):,}')
print('Muon matrices:', len(muon_parameter_names(model)))
display(pd.DataFrame([CONFIG.__dict__]))
del model


## Run or resume all nine reference trainings

Completed runs can be reloaded from their CSV files. Incomplete runs resume from `checkpoint_latest.pt`. For an infrastructure-only smoke test, create a temporary config with fewer epochs; do not report smoke-test metrics as baseline results.


In [ ]:
all_history = []
all_spectral = []
for optimizer_name in ('sgd_momentum', 'adamw', 'muon'):
    for seed in SEEDS:
        history, spectral = run_vit_baseline(
            optimizer_name, seed,
            data_dir=DATA_DIR, output_dir=EXPERIMENT_ROOT,
            config=CONFIG, device=DEVICE, progress=True, resume=True,
        )
        history.insert(0, 'seed', seed)
        history.insert(0, 'optimizer', optimizer_name)
        spectral.insert(0, 'seed', seed)
        spectral.insert(0, 'optimizer', optimizer_name)
        all_history.append(history)
        all_spectral.append(spectral)
performance = pd.concat(all_history, ignore_index=True)
spectral = pd.concat(all_spectral, ignore_index=True)
performance.to_csv(EXPERIMENT_ROOT / 'performance_all_runs.csv', index=False)
spectral.to_csv(EXPERIMENT_ROOT / 'weightwatcher_all_runs.csv', index=False)
final_summary = summarize_final(performance, CONFIG.epochs)
final_summary.to_csv(EXPERIMENT_ROOT / 'final_summary_95ci.csv', index=False)
display(final_summary)


## Performance trajectories with 95% Student-t intervals


In [ ]:
OPTIMIZER_COLORS = {'sgd_momentum':'#0072B2', 'adamw':'#D55E00', 'muon':'#009E73'}
OPTIMIZER_LABELS = {'sgd_momentum':'SGD + Nesterov', 'adamw':'AdamW', 'muon':'Muon + aux AdamW'}
TCRIT = 4.302652729911275

def plot_performance(metric):
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for optimizer_name, group in performance.groupby('optimizer'):
        for _, seed_group in group.groupby('seed'):
            ax.plot(seed_group['epoch'], seed_group[metric], color=OPTIMIZER_COLORS[optimizer_name], alpha=0.18, linewidth=0.8)
        stats = group.groupby('epoch')[metric].agg(['mean','std','count']).reset_index()
        half = TCRIT * stats['std'].fillna(0) / np.sqrt(stats['count'])
        ax.plot(stats['epoch'], stats['mean'], color=OPTIMIZER_COLORS[optimizer_name], label=OPTIMIZER_LABELS[optimizer_name])
        ax.fill_between(stats['epoch'], stats['mean']-half, stats['mean']+half, color=OPTIMIZER_COLORS[optimizer_name], alpha=0.16)
    ax.set_xlabel('Epoch'); ax.set_ylabel(metric.replace('_',' ')); ax.set_title(f'{metric}: mean and 95% Student-t CI')
    ax.grid(alpha=0.25); ax.legend(frameon=False); fig.tight_layout(); plt.show()

for metric in ('train_loss','validation_loss','test_loss','train_accuracy','validation_accuracy','test_accuracy'):
    plot_performance(metric)


## Learning-rate schedules


In [ ]:
for metric in ('primary_lr','auxiliary_lr'):
    fig, ax = plt.subplots(figsize=(9,4.5))
    for optimizer_name, group in performance.groupby('optimizer'):
        values = group.groupby('epoch')[metric].mean()
        if values.notna().any():
            ax.plot(values.index, values.values, label=OPTIMIZER_LABELS[optimizer_name], color=OPTIMIZER_COLORS[optimizer_name])
    ax.set_yscale('log'); ax.set_xlabel('Epoch'); ax.set_ylabel(metric); ax.grid(alpha=.25); ax.legend(frameon=False); fig.tight_layout(); plt.show()


## Layerwise alpha, ERG gap, and correlation traps

Matrix colors are invariant across optimizers and epochs.


In [ ]:
MATRIX_COLORS = {'W_Q':'#0072B2','W_K':'#E69F00','W_V':'#009E73','W_O':'#D55E00','W_MLP_IN':'#CC79A7','W_MLP_OUT':'#56B4E9'}
for optimizer_name in ('sgd_momentum','adamw','muon'):
    subset = spectral[spectral['optimizer'].eq(optimizer_name)]
    for metric in ('alpha','ERG_gap','num_traps'):
        fig, ax = plt.subplots(figsize=(11,6))
        grouped = subset.groupby(['matrix_type','epoch'])[metric].agg(['mean','std','count']).reset_index()
        for matrix_type, group in grouped.groupby('matrix_type'):
            half = TCRIT * group['std'].fillna(0) / np.sqrt(group['count'])
            color = MATRIX_COLORS[matrix_type]
            ax.plot(group['epoch'], group['mean'], color=color, label=matrix_type)
            ax.fill_between(group['epoch'], group['mean']-half, group['mean']+half, color=color, alpha=.12)
        if metric == 'alpha': ax.axhline(2.0, linestyle='--', linewidth=1)
        if metric == 'ERG_gap': ax.axhline(0.0, linestyle='--', linewidth=1)
        ax.set_xlabel('Epoch'); ax.set_ylabel(metric); ax.set_title(f'{OPTIMIZER_LABELS[optimizer_name]} — {metric}')
        ax.grid(alpha=.25); ax.legend(ncol=3, frameon=False); fig.tight_layout(); plt.show()


## Baseline contract

Validation loss selects the best checkpoint. Test curves are monitoring-only and never change the optimizer, schedule, stopping rule, or checkpoint selection. The full recipe audit is in `baseline/BASELINE_RECIPE_AUDIT.md`.
